In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.5.0


In [2]:
!python -V

Python 3.10.13


In [3]:
import pickle
import pandas as pd

In [4]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [5]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [10]:
year = 2023
month = 3

In [11]:
df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet')

In [12]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [13]:
print(type(y_pred))
print(y_pred.shape)
print(y_pred[:10])

<class 'numpy.ndarray'>
(3316216,)
[16.24590642 26.1347962  11.88426424 11.99771983 10.23448579 10.59717421
 12.44479314 10.972209   23.12034345 10.28111639]


In [14]:
import numpy as np
print(np.std(y_pred))

6.247488852238703


In [15]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [17]:
df['ride_id'].head(2)

0    2023/03_0
1    2023/03_1
Name: ride_id, dtype: object

In [18]:
def save_predictions(df, preds, year, month, output_file):
    # create ride_id
    df["ride_id"] = f"{year:04d}/{month:02d}_" + df.index.astype(str)

    # assemble results
    df_result = pd.DataFrame({
        "ride_id": df["ride_id"],
        "predicted_duration": preds
    })

    # write to Parquet
    df_result.to_parquet(
        output_file,
        engine="pyarrow",
        compression=None,
        index=False
    )

In [19]:
output_file = f'yellow_tripdata_{year:04d}-{month:02d}_predictions.parquet'
save_predictions(df, y_pred, year, month, output_file)